## Task1: Tối ưu hóa hàm số với thuật toán Hill Climbing

**Môn học:** Trí tuệ nhân tạo

**Phương pháp:** Lập trình hướng đối tượng (OOP)

**1. YÊU CẦU BÀI TOÁN**

---

**Hàm số $f(x,y)$ cần tối ưu:**
$$
f(x, y) = \sin\left(\frac{x}{4}\right) + \cos\left(\frac{y}{4}\right) - \sin\left(\frac{x \cdot y}{16}\right) + \cos\left(\frac{5x^2}{16}\right) + \sin\left(\frac{5y^2}{16}\right)
$$

**1.1 Yêu cầu 1 (0.5 điểm)**

Vẽ mặt phẳng 3D minh họa hàm $f$ và đường đi của thuật toán.


---

**1.2 Yêu cầu 2 (2.0 điểm)**

Cài đặt thuật toán Hill Climbing để tìm vị trí $(x, y)$ có giá trị $f$ lớn nhất, bắt đầu từ $O(0, 0)$.

**Các ý cần giải quyết:**

1.  Đề xuất cách xác định lân cận tốt nhất ngay lập tức với độ phức tạp $O(1)$

2.  Đề xuất cách kiểm soát kích thước bước nhảy (khoảng cách giữa trạng thái hiện tại và lân cận được chọn).

3.  Đề xuất cách xử lý trường hợp không còn lân cận nào tốt hơn (kẹt tại cực đại cục bộ).

---

**1.3 Yêu cầu 3 (0.5 điểm)**

Tổ chức chương trình theo mô hình Hướng đối tượng (OOP), đảm bảo mã nguồn ngắn gọn và hợp lý.

**2. TÓM TẮT TRIỂN KHAI THUẬT TOÁN HILL CLIMBING (HC)**

---

Việc triển khai thuật toán **Hill Climbing** được thực hiện theo mô hình **Lập trình Hướng đối tượng (OOP)**, sử dụng chiến lược tìm kiếm lân cận **Multi-Step Radius Search** để cải thiện khả năng thoát khỏi các cực đại cục bộ.

---

**2.1 Cấu Trúc OOP (4 Lớp)**

| Lớp (Class) | Chức năng Chính | Chi tiết |
| :--- | :--- | :--- |
| **`State`** | **Đại diện trạng thái** và sinh lân cận. | Lưu trữ tọa độ $(x, y)$. Thiết kế để **sinh 8 lân cận** (lên, xuống, trái, phải, chéo) với độ phức tạp $O(1)$. |
| **`Problem`** | **Định nghĩa hàm mục tiêu.** | Sử dụng thư viện **SymPy** để tạo hàm $f(x, y)$ và **`lambdify`** để tối ưu tốc độ tính toán. |
| **`HillClimbing`** | **Cài đặt logic thuật toán.** | Thực thi vòng lặp chính của HC, áp dụng chiến lược **Multi-Step Radius Search** với danh sách kích thước bước nhảy (ví dụ: $[0.1, 0.3, 0.7]$). |
| **`Visualizer`** | **Trực quan hóa kết quả.** | Vẽ **đồ thị 3D** của hàm số, minh họa **đường đi** của thuật toán bằng đường màu đỏ, và đánh dấu điểm bắt đầu/kết thúc.  |

---

**2.2 Chiến Lược Multi-Step Radius Search**

**Khái niệm:** **Multi-Step Radius Search** (Tìm kiếm Bán kính Đa Bước) là một biến thể nâng cao của Hill Climbing, trong đó thuật toán không chỉ xét các lân cận gần nhất (Nearest Neighbors) mà còn thăm dò đồng thời các lân cận ở nhiều khoảng cách (bán kính) khác nhau. Chiến lược này giúp tăng cường khả năng **khám phá không gian** và hiệu quả hơn trong việc **thoát khỏi các cực đại cục bộ** nhỏ hoặc các vùng cao nguyên (plateaus) bằng cách cung cấp các "bước nhảy lớn hơn" một cách có chọn lọc.

Chiến lược này nhằm mục đích **khám phá không gian tìm kiếm rộng hơn** trong mỗi bước nhảy, giảm nguy cơ bị kẹt tại các cực đại cục bộ nhỏ.

***Cách thức hoạt động:***

1.  **Thăm dò Đa Bán kính:** Thay vì chỉ dùng một bước nhảy cố định, thuật toán sử dụng một danh sách các bán kính **`radius_list`** (ví dụ: $[0.1, 0.3, 0.7]$). Tại mỗi bước, nó sinh lân cận cho **từng** bán kính trong danh sách.
2.  **Tạo Super-Neighborhood:** Tất cả lân cận được sinh ra từ các bán kính khác nhau sẽ được **gom lại** thành một tập hợp lớn (Super-Neighborhood).
3.  **Tìm kiếm Tốt nhất (Steepest-Ascent):** Thuật toán đánh giá tất cả các điểm trong Super-Neighborhood và **chọn lân cận có giá trị $f$ lớn nhất** để làm điểm di chuyển tiếp theo.
4.  **Di chuyển và Dừng:**
* **Di chuyển** nếu lân cận tốt nhất có giá trị $f$ cao hơn trạng thái hiện tại.
    * **Dừng lại** nếu không tìm thấy bất kỳ lân cận nào (trong bất kỳ bán kính nào) có giá trị $f$ tốt hơn, cho thấy thuật toán đã tìm thấy cực đại trong phạm vi đã xét.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sympy import symbols, sin, cos
from sympy.utilities.lambdify import lambdify

# ======================================================
# 1. CLASS STATE
# ======================================================
class State:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def get_neighbors(self, step_size):
        """Sinh 8 láng giềng – O(1)."""
        directions = [
            (step_size, 0), (-step_size, 0),
            (0, step_size), (0, -step_size),
            (step_size, step_size), (step_size, -step_size),
            (-step_size, step_size), (-step_size, -step_size)
        ]
        return [State(self.x + dx, self.y + dy) for dx, dy in directions]

    def __repr__(self):
        return f"State({self.x:.2f}, {self.y:.2f})"


# ======================================================
# 2. CLASS PROBLEM (KHÔNG HARD-CODE HÀM f)
# ======================================================
class Problem:
    def __init__(self, f_expr):
        x, y = symbols("x y")
        self.f_sym = f_expr
        self.f = lambdify((x, y), self.f_sym, "numpy")

    def value(self, s: State):
        return self.f(s.x, s.y)


# ======================================================
# 3. MULTI-STEP RADIUS HILL CLIMBING
# ======================================================
class HillClimbing:
    def __init__(self, radius_list, max_iter=1000):
        self.radius_list = radius_list
        self.max_iter = max_iter

    def solve(self, problem: Problem, initial_state: State):
        current = initial_state
        path = [current]
        print(f"Bắt đầu tại: ({current.x:.4f}, {current.y:.4f}), f = {problem.value(current):.5f}")

        for _ in range(self.max_iter):
            current_val = problem.value(current)

            # MULTI-STEP search
            neighbors = []
            for r in self.radius_list:
                neighbors.extend(current.get_neighbors(step_size=r))

            # chọn hàng xóm tốt nhất
            best_neighbor = max(neighbors, key=lambda s: problem.value(s))
            best_val = problem.value(best_neighbor)

            # nếu không có hàng xóm tốt hơn: dừng
            if best_val <= current_val:
                print(f"--- Dừng tại Bước {_+1}: Không tìm thấy lân cận tốt hơn (Cực đại cục bộ).")
                break

            current = best_neighbor
            path.append(current)
            print(f"Bước {_+1}: Di chuyển đến ({current.x:.4f}, {current.y:.4f}), f = {best_val:.5f}")

        return current, path


# ======================================================
# 4. VISUALIZER
# ======================================================
class Visualizer:
    def __init__(self, problem: Problem):
        self.problem = problem

    def draw(self, path):
        x = np.linspace(-10, 10, 150)
        y = np.linspace(-10, 10, 150)
        X, Y = np.meshgrid(x, y)
        Z = self.problem.f(X, Y)

        px = [s.x for s in path]
        py = [s.y for s in path]
        pz = [self.problem.value(s) for s in path]

        fig = plt.figure(figsize=(12, 6))
        ax = fig.add_subplot(1, 2, 1, projection='3d')

        ax.plot_surface(X, Y, Z, cmap="viridis", alpha=0.7)
        ax.plot(px, py, pz, color="red", linewidth=3, marker="o")
        ax.scatter(px[0], py[0], pz[0], color="yellow", s=100, label="Start")
        ax.scatter(px[-1], py[-1], pz[-1], color="black", s=120, marker="*", label="End")

        ax.set_title("Mô hình 3D + đường đi tìm kiếm (Hill Climbing)")
        ax.legend()

        # 2D heatmap
        ax2 = fig.add_subplot(1, 2, 2)
        contour = ax2.contourf(X, Y, Z, 50, cmap="viridis")
        plt.colorbar(contour)
        ax2.plot(px, py, color="red", marker="o")
        ax2.scatter(px[0], py[0], color="white", marker="x", s=100)
        ax2.scatter(px[-1], py[-1], color="black", marker="*", s=150)
        ax2.set_title("Đường đi tìm kiếm bằng mô hình 2D")

        plt.tight_layout()
        plt.show()


# ======================================================
# 5. MAIN — TRUYỀN HÀM f TỪ BÊN NGOÀI
# ======================================================
if __name__ == "__main__":
    x, y = symbols("x y")

    # Hàm f ĐƯỢC ĐỊNH NGHĨA BÊN NGOÀI — KHÔNG GÁN CỨNG TRONG CLASS
    f_expr = (
        sin(x/4) + cos(y/4)
        - sin(x*y/16)
        + cos(x**2/16)
        + sin(y**2/16)
    )

    problem = Problem(f_expr)

    start = State(0, 0)
    hill = HillClimbing(radius_list=[0.1, 0.3, 0.7])

    final_state, path = hill.solve(problem, start)

    print("\n===== KẾT QUẢ =====")
    print("Điểm cực đại:", final_state)
    print("Giá trị cực đại f =", problem.value(final_state))
    print("Tổng số bước di chuyển:", len(path) - 1)

    viz = Visualizer(problem)
    viz.draw(path)
